# Reto Chimera — Grupo 77 · corrida completa en Colab

Corre las **tres etapas** (arquitectura propia → transfer → fine-tuning) sobre una
GPU T4 y deja listos los entregables.

### Cómo usarlo

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. Sin esto no arranca.
2. **Entorno de ejecución → Ejecutar todas**.
3. **No cierres esta pestaña.** Colab desconecta las sesiones sin pestaña abierta y
   se pierde el entrenamiento.

Tiempo estimado en T4: **45–75 min** (Intel ~5 min · transfer ~25 min · fine-tuning ~20 min),
más ~10 min de descarga de datasets.

Al final la última celda descarga `grupo77_entrega.zip` con los tres `.pth`,
el `comprobante.json` y el notebook ejecutado.

> **Los batch size no se tocan** (32 en Intel, 16 en PathMNIST). El `fingerprint_sha256`
> de cada checkpoint se calcula sobre el primer batch del test set: si cambia el tamaño
> del batch, el hash deja de coincidir cuando el profesor lo recalcule.

## 1. Comprobar que hay GPU

In [ ]:
import subprocess, sys

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi no existe: este entorno NO tiene GPU.")

import torch
print("torch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "\n>>> SIN GPU. Ve a: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4), "
        "y vuelve a ejecutar todo. En CPU esto tarda mas de un dia."
    )
print("\nGPU:", torch.cuda.get_device_name(0),
      f"| VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Repo y dependencias

In [ ]:
%cd /content
# Si ya esta clonado NO se hace pull: el notebook local puede tener resultados
# de una corrida a medias y un pull los sobrescribiria.
!if [ -d reto-chimera ]; then echo "ya existe /content/reto-chimera, se conserva el estado local"; else git clone -q https://github.com/caesar-dat-com/reto-chimera; fi
%cd /content/reto-chimera

# Colab ya trae torch, numpy, sklearn, pillow, requests. Solo faltan estos.
!pip install -q nbclient nbformat

import os
print("\nArchivos:", sorted(os.listdir(".")))

## 3. Datasets

Intel (~363 MB desde HuggingFace) y PathMNIST (~1 GB desde Zenodo). Se arman como
`ImageFolder` en `./data/`. Si la celda se reejecuta, reusa lo ya descargado.

In [ ]:
!python -u scripts/prepare_intel_dataset.py
print("\n" + "=" * 70 + "\n")
!python -u scripts/prepare_pathmnist_dataset.py

In [ ]:
# Verificacion: conteos reales en disco antes de gastar una hora de GPU
import os

for nombre, ruta in [("Intel", "data/intel_subset"), ("PathMNIST", "data/pathmnist_subset")]:
    if not os.path.isdir(ruta):
        raise SystemExit(f"FALTA {ruta} -- reejecuta la celda anterior")
    print(f"{nombre}  ({ruta})")
    for split in ("train", "test"):
        base = os.path.join(ruta, split)
        clases = sorted(os.listdir(base))
        total = sum(len(os.listdir(os.path.join(base, c))) for c in clases)
        print(f"  {split:<6} {total:>6} imagenes en {len(clases)} clases")

## 4. Ajuste del notebook para GPU

El notebook viene con `NUM_WORKERS = 0` y `PIN_MEMORY = False` porque se escribió para
correr en un portátil sin GPU. En Colab eso deja la T4 esperando al data loader.

Este parche cambia **solo** esas dos constantes y activa `cudnn.benchmark`. No toca
batch sizes, épocas, arquitectura ni semilla.

In [ ]:
import nbformat

NB = "S3_Reto_Hibridacion_Chimera_tester.ipynb"
nb = nbformat.read(NB, as_version=4)

cambios = []
for celda in nb.cells:
    if celda.cell_type != "code":
        continue
    src = celda.source
    if "NUM_WORKERS = 0" in src:
        src = src.replace("NUM_WORKERS = 0", "NUM_WORKERS = 2")
        cambios.append("NUM_WORKERS -> 2")
    if "PIN_MEMORY = False" in src:
        src = src.replace("PIN_MEMORY = False", "PIN_MEMORY = True")
        cambios.append("PIN_MEMORY -> True")
    if "device = torch.device(" in src and "cudnn.benchmark" not in src:
        src = src.replace(
            'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")',
            'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
            'torch.backends.cudnn.benchmark = True  # Colab: elige el kernel de conv mas rapido',
        )
        cambios.append("cudnn.benchmark -> True")
    celda.source = src

nbformat.write(nb, open(NB, "w", encoding="utf-8"))
print("Parches aplicados:", cambios or "ninguno (ya estaba parcheado)")

# Confirmacion de que la config del grupo sigue intacta
fuente = "".join(c.source for c in nb.cells if c.cell_type == "code")
for esperado in ["CODIGO_GRUPO = 77", "BATCH_SIZE = 32", "BATCH_SIZE_PATH = 16"]:
    print(("OK  " if esperado in fuente else "!!! FALTA ") + esperado)

## 5. Entrenamiento completo

Las tres etapas van en **un solo kernel** a propósito: transfer y fine-tuning parten
del `modelo_propio` que vive en memoria tras entrenar en Intel.

`ejecutar_notebook.py` guarda el `.ipynb` **después de cada celda**, así que si algo
revienta a mitad de camino queda todo lo que sí corrió.

Umbrales del profesor: base F1≥0.80 (3.0) / ≥0.95 (5.0) · transfer ≥0.77 / ≥0.85 ·
fine-tuning ≥0.78 / ≥0.87.

In [ ]:
import time
t0 = time.time()
!python -u scripts/ejecutar_notebook.py
print(f"\n=== Celda terminada en {(time.time()-t0)/60:.1f} min ===")

## 6. Verificar entregables y limpiar salidas ajenas

El `.ipynb` original venía ejecutado por el profesor (checkpoints `grupo0_*.pth`).
Si alguna celda no llegó a correr aquí, su salida seguiría siendo la de él. Esta celda
compara contra la versión de git y vacía cualquier salida que no sea de esta corrida.

In [ ]:
import json, os, subprocess
import nbformat

NB = "S3_Reto_Hibridacion_Chimera_tester.ipynb"

original = nbformat.reads(
    subprocess.run(["git", "show", f"HEAD:{NB}"], capture_output=True, text=True).stdout,
    as_version=4,
)
actual = nbformat.read(NB, as_version=4)

ajenas = []
for i, (c_orig, c_act) in enumerate(zip(original.cells, actual.cells)):
    if c_act.cell_type != "code" or not c_act.get("outputs"):
        continue
    # Salida byte a byte identica a la del repo = esta celda no se ejecuto aqui
    if c_orig.get("outputs") and c_orig["outputs"] == c_act["outputs"]:
        c_act["outputs"] = []
        c_act["execution_count"] = None
        ajenas.append(i)

if ajenas:
    print(f"Salidas del profesor borradas en las celdas: {ajenas}")
    print(">>> La corrida quedo INCOMPLETA. Revisa el error de la celda 5.")
else:
    print("Todas las celdas de codigo tienen salidas propias de esta corrida.")

nbformat.write(actual, open(NB, "w", encoding="utf-8"))

print("\nEntregables en ./entregas/:")
esperados = ["grupo77_arquitectura_propia.pth", "grupo77_transfer.pth",
             "grupo77_finetune.pth", "grupo77_comprobante.json"]
completo = True
for n in esperados:
    ruta = os.path.join("entregas", n)
    if os.path.exists(ruta):
        print(f"  OK      {n}  ({os.path.getsize(ruta)/1e6:.1f} MB)")
    else:
        print(f"  FALTA   {n}")
        completo = False

if os.path.exists("entregas/grupo77_comprobante.json"):
    print("\n--- Resultados ---")
    print(json.dumps(json.load(open("entregas/grupo77_comprobante.json")), indent=2, ensure_ascii=False))

## 7. Empaquetar y descargar

Genera `grupo77_entrega.zip` y lo descarga al computador. **Guárdalo antes de cerrar
Colab**: el disco de la sesión se borra al desconectar.

In [ ]:
import os, shutil, zipfile

NB = "S3_Reto_Hibridacion_Chimera_tester.ipynb"
os.makedirs("entregas", exist_ok=True)
shutil.copy(NB, "entregas/grupo77_notebook_ejecutado.ipynb")

ZIP = "/content/grupo77_entrega.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for n in sorted(os.listdir("entregas")):
        z.write(os.path.join("entregas", n), n)
    z.write("chimera_blocks.py", "chimera_blocks.py")

print(f"{ZIP}  ({os.path.getsize(ZIP)/1e6:.1f} MB)")
for n in zipfile.ZipFile(ZIP).namelist():
    print("  -", n)

from google.colab import files
files.download(ZIP)

### Respaldo opcional en Drive

Si la sesión se va a alargar, esta celda deja una copia en `MiUnidad/reto-chimera/`
para no depender de la descarga.

In [ ]:
# Descomenta las tres lineas si quieres respaldo en Drive
# from google.colab import drive; drive.mount("/content/drive")
# import shutil, os; os.makedirs("/content/drive/MyDrive/reto-chimera", exist_ok=True)
# shutil.copy("/content/grupo77_entrega.zip", "/content/drive/MyDrive/reto-chimera/")